In [2]:
%load_ext autoreload
%autoreload 2
import helper_functions as hf
from imports import *
import importlib

num_available_cpus = multiprocessing.cpu_count()
print("Number of available CPUs:", num_available_cpus)

torch.cuda.empty_cache()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device =", device)
torch.set_default_tensor_type('torch.cuda.FloatTensor') if torch.cuda.is_available() else print ('cpu')

torch.set_num_threads(num_available_cpus)

print("Number of threads:", torch.get_num_threads())
print("Number of interop threads:", torch.get_num_interop_threads())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Number of available CPUs: 80
Device = cuda:0
Number of threads: 80
Number of interop threads: 80


In [6]:
def chunker(a,i,nbin):
    numAx = a.shape[1]
    ind = np.argsort(a[:,i])
    a = a[ind]
    if i+1 == numAx:
        return np.array_split(a,nbin,axis=0)
    else:
        return [chunker(subA,i+1,nbin) for subA in np.array_split(a,nbin,axis=0)]
    
def makeMassPlots(baseDir,year):
    from matplotlib.backends.backend_pdf import PdfPages
    spaces = ["QCDBKG_massFlat_2017-Qstar2000_W400_UL17-and-Wp3000_B400_UL17-and-XYY_X3000_Y80_UL17"]

    for space in spaces:
        axes =["QCDBKG_massFlat_2017","Qstar2000_W400_UL17-and-Wp3000_B400_UL17-and-XYY_X3000_Y80_UL17"]
        data_ax = axes[0]
        other_ax = axes[1:]
        nAx = len(axes)
        if nAx == 2:
            bins_per = 5
        if nAx == 3:
            bins_per = 4
        if nAx == 4:
            bins_per = 3

        bkgs = [baseDir+space+"/"+f for f in os.listdir(baseDir+space) if "eval_QCDBKG" in f]

        loss_by_ax = {ax:[] for ax in axes}
        masses = []
        for bfile in bkgs:
            with h5py.File(bfile,"r") as f:
                for ax in axes:
                    loss_by_ax[ax].append(f[ax][()])
                masses.append(f['mass'][()])
        masses = np.concatenate(masses)
        for ax in axes:
            loss_by_ax[ax] = np.concatenate(loss_by_ax[ax])

        total = [masses.reshape(-1,1)] + [loss_by_ax[data_ax].reshape(-1,1)] + [loss_by_ax[ax].reshape(-1,1) for ax in other_ax]
        total = np.concatenate(total,axis=1)
        total = total[np.argsort(total[:,1])]
        total = total[int(0.9*total.shape[0]):]
        chunked = chunker(total,1,bins_per)

        mass_bins = np.arange(1800,6100,step=100)
        axis_order = [data_ax] + other_ax

        if nAx == 2:
            fig,axes = plt.subplots(bins_per,bins_per,figsize=(20,20))
            fig.supxlabel(r"QCD MC Loss Bin $\longrightarrow$".format(axis_order[1]),fontsize=16)
            fig.supylabel(r"Mixed Signal Loss Bin $\longrightarrow$".format(axis_order[0]),fontsize=16)
            for i in range(bins_per):
                for j in range(bins_per):
                    bin_mass = chunked[i][j][:,0]
                    h = axes[bins_per-j-1,i].hist(bin_mass,bins=mass_bins,density=True)
                    axes[bins_per-j-1,i].ticklabel_format(axis='y',style='sci',scilimits=(-3,-3))
            plt.tight_layout()
            plt.savefig(baseDir+space+"/binned_mjj_{0}.pdf".format(("{0}by".format(bins_per)*nAx)[:-2]))
            plt.close()
        if nAx == 3:
            fig,axes = plt.subplots(bins_per,bins_per,figsize=(20,20))
            fig.supxlabel(r"{0} Loss Bin $\longrightarrow$".format(axis_order[1]),fontsize=16)
            fig.supylabel(r"$\longleftarrow$ {0} Loss Bin".format(axis_order[0]),fontsize=16)
            for i in range(bins_per):
                for j in range(bins_per):
                    for k in range(bins_per):
                        bin_mass = chunked[i][j][k][:,0]
                        h = axes[i,j].hist(bin_mass,bins=mass_bins,density=True,histtype='step',label='{0} Bin {1}'.format(axis_order[2],k+1))
                    axes[i,j].legend(loc='upper right')
                    axes[i,j].ticklabel_format(axis='y',style='sci',scilimits=(-3,-3))
            plt.tight_layout(rect=[0.02,0.02,0.98,0.98])
            plt.savefig(baseDir+space+"/binned_mjj_{0}.pdf".format(("{0}by".format(bins_per)*nAx)[:-2]))
            plt.close()
        if nAx == 4:
            with PdfPages(baseDir+space+"/binned_mjj_{0}.pdf".format(("{0}by".format(bins_per)*nAx)[:-2])) as pdfpage:
                for ii in range(bins_per):
                    fig,axes = plt.subplots(bins_per,bins_per,figsize=(20,20))
                    fig.suptitle("dataSB bin {0}".format(ii),fontsize=20)
                    fig.supxlabel(r"{0} Loss Bin $\longrightarrow$".format(axis_order[2]),fontsize=16)
                    fig.supylabel(r"$\longleftarrow$ {0} Loss Bin".format(axis_order[1]),fontsize=16)
                    for i in range(bins_per):
                        for j in range(bins_per):
                            for k in range(bins_per):
                                bin_mass = chunked[ii][i][j][k][:,0]
                                h = axes[i,j].hist(bin_mass,bins=mass_bins,density=True,histtype='step',label='{0} Bin {1}'.format(axis_order[3],k+1))
                            axes[i,j].legend(loc='upper right')
                            axes[i,j].ticklabel_format(axis='y',style='sci',scilimits=(-3,-3))
                    plt.tight_layout(rect=[0.02,0.02,0.98,0.98])
                    pdfpage.savefig()
                    plt.close()

In [7]:
baseDir = "evaluations/qcdbkg_massFlat/2017/quakSpaces/"
makeMassPlots(baseDir,2017)

In [13]:
baseDir = "evaluations/dataSB_massFlat/2017/quakSpaces/"
makeMassPlots(baseDir,2017)

In [14]:
baseDir = "evaluations/dataSB/2017/quakSpaces/"
makeMassPlots(baseDir,2017)

In [15]:
baseDir = "evaluations/qcdbkg_massFlat/2017/quakSpaces/"
makeMassPlots(baseDir,2017)

In [16]:
baseDir = "evaluations/qcdbkg/2017/quakSpaces/"
makeMassPlots(baseDir,2017)

In [24]:
# Checking for sculpting in bkg trainings done with data sideband
baseDir = "evaluations/dataSB/quakSpaces/"
spaces = [d for d in os.listdir(baseDir) if 'dataSB' in d]
print(spaces)

['dataSB-XYY_X3000_Y80_UL17-Qstar2000_W400_UL17', 'dataSB-Wp3000_B400_UL17', 'dataSB-Qstar2000_W400_UL17', 'dataSB-XYY_X3000_Y80_UL17-Wp3000_B400_UL17', 'dataSB-Wp3000_B400_UL17-Qstar2000_W400_UL17', 'dataSB-XYY_X3000_Y80_UL17', 'dataSB-XYY_X3000_Y80_UL17-Wp3000_B400_UL17-Qstar2000_W400_UL17']


In [25]:
from matplotlib.backends.backend_pdf import PdfPages

for space in spaces:
    axes = space.split('-')
    data_ax = 'dataSB'
    other_ax = [a for a in axes if a != 'dataSB']
    nAx = len(axes)
    if nAx == 2:
        bins_per = 9
    if nAx == 3:
        bins_per = 4
    if nAx == 4:
        bins_per = 3
    
    bkgs = [baseDir+space+"/"+f for f in os.listdir(baseDir+space) if "eval_QCDBKG" in f]
    
    loss_by_ax = {ax:[] for ax in axes}
    masses = []
    for bfile in bkgs:
        with h5py.File(bfile,"r") as f:
            for ax in axes:
                loss_by_ax[ax].append(f[ax][()])
            masses.append(f['mass'][()])
    masses = np.concatenate(masses)
    for ax in axes:
        loss_by_ax[ax] = np.concatenate(loss_by_ax[ax])
    
    total = [masses.reshape(-1,1)] + [loss_by_ax['dataSB'].reshape(-1,1)] + [loss_by_ax[ax].reshape(-1,1) for ax in other_ax]
    total = np.concatenate(total,axis=1)
    total = total[np.argsort(total[:,1])]
    total = total[int(0.9*total.shape[0]):]
    chunked = chunker(total,1,bins_per)
    
    mass_bins = np.linspace(1800,5000,num=100)
    axis_order = ['dataSB'] + other_ax
    
    if nAx == 2:
        fig,axes = plt.subplots(bins_per,bins_per,figsize=(20,20))
        fig.supxlabel(r"{0} Loss Bin $\longrightarrow$".format(axis_order[1]),fontsize=16)
        fig.supylabel(r"$\longleftarrow$ {0} Loss Bin".format(axis_order[0]),fontsize=16)
        for i in range(bins_per):
            for j in range(bins_per):
                bin_mass = chunked[i][j][:,0]
                h = axes[i,j].hist(bin_mass,bins=mass_bins,density=True)
                axes[i,j].ticklabel_format(axis='y',style='sci',scilimits=(-3,-3))
        plt.tight_layout()
        plt.savefig(baseDir+space+"/binned_mjj_{0}.pdf".format(("{0}by".format(bins_per)*nAx)[:-2]))
        plt.close()
    if nAx == 3:
        fig,axes = plt.subplots(bins_per,bins_per,figsize=(20,20))
        fig.supxlabel(r"{0} Loss Bin $\longrightarrow$".format(axis_order[1]),fontsize=16)
        fig.supylabel(r"$\longleftarrow$ {0} Loss Bin".format(axis_order[0]),fontsize=16)
        for i in range(bins_per):
            for j in range(bins_per):
                for k in range(bins_per):
                    bin_mass = chunked[i][j][k][:,0]
                    h = axes[i,j].hist(bin_mass,bins=mass_bins,density=True,histtype='step',label='{0} Bin {1}'.format(axis_order[2],k+1))
                axes[i,j].legend(loc='upper right')
                axes[i,j].ticklabel_format(axis='y',style='sci',scilimits=(-3,-3))
        plt.tight_layout(rect=[0.02,0.02,0.98,0.98])
        plt.savefig(baseDir+space+"/binned_mjj_{0}.pdf".format(("{0}by".format(bins_per)*nAx)[:-2]))
        plt.close()
    if nAx == 4:
        with PdfPages(baseDir+space+"/binned_mjj_{0}.pdf".format(("{0}by".format(bins_per)*nAx)[:-2])) as pdfpage:
            for ii in range(bins_per):
                fig,axes = plt.subplots(bins_per,bins_per,figsize=(20,20))
                fig.suptitle("dataSB bin {0}".format(ii),fontsize=20)
                fig.supxlabel(r"{0} Loss Bin $\longrightarrow$".format(axis_order[2]),fontsize=16)
                fig.supylabel(r"$\longleftarrow$ {0} Loss Bin".format(axis_order[1]),fontsize=16)
                for i in range(bins_per):
                    for j in range(bins_per):
                        for k in range(bins_per):
                            bin_mass = chunked[ii][i][j][k][:,0]
                            h = axes[i,j].hist(bin_mass,bins=mass_bins,density=True,histtype='step',label='{0} Bin {1}'.format(axis_order[3],k+1))
                        axes[i,j].legend(loc='upper right')
                        axes[i,j].ticklabel_format(axis='y',style='sci',scilimits=(-3,-3))
                plt.tight_layout(rect=[0.02,0.02,0.98,0.98])
                pdfpage.savefig()
                plt.close()

In [71]:
a = np.random.randn(8,2)
print(a)
f = chunker(a,0,2)
f

[[ 0.05672659 -1.33631025]
 [ 0.50722713  0.67865929]
 [ 0.05178463  0.31065027]
 [-0.31736861 -1.30799696]
 [-0.51769897  0.72875131]
 [-2.05159042 -0.15915056]
 [ 0.95918309  0.26987561]
 [-0.94654205 -1.03412266]]


[[array([[-0.31736861, -1.30799696],
         [-0.94654205, -1.03412266]]),
  array([[-2.05159042, -0.15915056],
         [-0.51769897,  0.72875131]])],
 [array([[ 0.05672659, -1.33631025],
         [ 0.95918309,  0.26987561]]),
  array([[0.05178463, 0.31065027],
         [0.50722713, 0.67865929]])]]